# Model Tester

In [30]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [31]:
import os
import sys
import math
import logging
import healpy as hp
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("/users/stevensonb/Research/tools/deepsphere-cosmo-tf2")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf

tf.get_logger().setLevel(logging.ERROR)

from tensorflow.keras.layers import Dense, Dropout, Flatten, LeakyReLU, Add
from tensorflow.keras.callbacks import EarlyStopping, TerminateOnNaN, TensorBoard
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.optimizers.schedules import CosineDecayRestarts, ExponentialDecay

from deepsphere import HealpyGCNN
from deepsphere.healpy_layers import (
    HealpyChebyshev,
    HealpyPool,
    Healpy_ResidualLayer,
    HealpyPseudoConv_Transpose,
)

from mlpng import Core
from mlpng.utils import setup_logging, try_init_wandb, make_trainer_plots
from mlpng.utils.dataloaders import MapDataset, UnlensMapDataset, UnlensPhiMapDataset
from mlpng.utils.callbacks import RMSELoss, RMSELoss2, rmse_metrics

logger = setup_logging("mlpng.trainer", level=logging.DEBUG)

In [32]:
print("Conda environment:", os.environ["CONDA_DEFAULT_ENV"])
print("Python executable:", sys.executable)
print(f"TensorFlow version: {tf.__version__}")
print(f"CUDA version: {tf.sysconfig.get_build_info()['cuda_version']}")
print(f"cuDNN version: {tf.sysconfig.get_build_info()['cudnn_version']}")

Conda environment: ds25
Python executable: /users/stevensonb/.conda/envs/ds25/bin/python
TensorFlow version: 2.15.1
CUDA version: 12.2
cuDNN version: 8


In [33]:
core = Core(
    [
        "settings/n256.json",
        "--nsims",
        "10000",
        "--phi_scale",
        "1",
        "--shapes",
        "local",
        "--fnl_range",
        "-1000",
        "1000",
    ]
)

shapes = core.shapes

03-Nov-25 18:18:55 - mlpng.core - DEBUG - Parsing CLI args: ['settings/n256.json', '--nsims', '10000', '--phi_scale', '1', '--shapes', 'local', '--fnl_range', '-1000', '1000']
03-Nov-25 18:18:55 - mlpng.core - INFO - Loading settings from file 'settings/n256.json'
03-Nov-25 18:18:55 - mlpng.core - DEBUG - Forcing setting 'nsims' to 10000 due to CLI
03-Nov-25 18:18:55 - mlpng.core - DEBUG - Forcing setting 'fnl_range' to [-1000, 1000] due to CLI
03-Nov-25 18:18:55 - mlpng.core - DEBUG - Forcing setting 'phi_scale' to 1.0 due to CLI
03-Nov-25 18:18:55 - mlpng.core - DEBUG - Forcing setting 'shapes' to ['local'] due to CLI
03-Nov-25 18:18:55 - mlpng.core - DEBUG - Overriding cosmo param 'As' from 2.13e-09 to 2.105e-09
03-Nov-25 18:18:55 - mlpng.core - DEBUG - Overriding cosmo param 'ns' from 0.9624 to 0.965
03-Nov-25 18:18:55 - mlpng.core - DEBUG - Running with settings: 
{
  "cosmo_params": {
    "As": 2.105e-09,
    "ns": 0.965,
    "pivot_scalar": 0.05,
    "H0": 67.4,
    "ombh2": 0.0

In [34]:
batch_size = 64  # TODO, figure out why chaning this breaks some of the data gen
max_epochs = 50
initial_LR = 5e-4

data_fraction = 0.1

unet_split = np.array([0.8, 0.1, 0.1]) * data_fraction
unet_duplicates = [25, 10, 2]

# fnl values come from the
fnl_split = np.array([0.4, 0.1, 0.5]) * data_fraction
fnl_duplicates = [25, 10, 2]

final_split = np.array([0.8, 0.1, 0.1]) * data_fraction
final_duplicates = [1, 1, 1]

strategy = tf.distribute.MirroredStrategy()  # use mirrored strategy for multi-GPU

In [35]:
# setup some common save file paths and cache paths

date_time = tf.timestamp().numpy().astype(int)
# date_time = "1761838112"  # freeze to one run to reuse the cache
run_name = f"notebook-{date_time}"
print("Run name:", run_name)

save_dir = f"{core.dirs['model']}/{core.name}"
run_info = f"{core.shapes_str()}-{run_name}"

# setups where we will
unet_keras_file = f"{save_dir}/unet-{run_info}.keras"
fnl_keras_file = f"{save_dir}/fnl-{run_info}.keras"
final_keras_file = f"{save_dir}/final-{run_info}.keras"
full_keras_file = f"{save_dir}/full-{run_info}.keras"

unet_cache = f"{core.name}/unet-{run_info}"
fnl_cache = f"{core.name}/fnl-{run_info}"
final_cache = f"{core.name}/final-{run_info}"

if os.path.exists(save_dir) is False:
    os.makedirs(save_dir, exist_ok=True)
    print(f"Created directory: {save_dir}")

for file in [
    unet_keras_file,
    fnl_keras_file,
    final_keras_file,
    full_keras_file,
]:
    print(f"file {file} exists: {os.path.exists(file)}")

Run name: notebook-1762215536
file data/models/l767_n256_T_10000_p1.0/unet-local-notebook-1762215536.keras exists: False
file data/models/l767_n256_T_10000_p1.0/fnl-local-notebook-1762215536.keras exists: False
file data/models/l767_n256_T_10000_p1.0/final-local-notebook-1762215536.keras exists: False
file data/models/l767_n256_T_10000_p1.0/full-local-notebook-1762215536.keras exists: False


In [36]:
# create the callbacks
callbacks = [
    TerminateOnNaN(),
    EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    tf.keras.callbacks.BackupAndRestore(
        f"{core.dirs['model']}/chkpts",
        save_freq="epoch",
        delete_checkpoint=True,
    ),
]

# if wanted we create a tensorboard and wandb callback, use the CLI to set these
tb_dir = f"{core.dirs['tb']}/{core.name}/{core.slurm.job}"
if core.use_tb:
    callbacks.append(
        TensorBoard(
            log_dir=tb_dir,
            histogram_freq=1,
            write_steps_per_second=True,
        )
    )

if core.use_wandb:
    try_init_wandb(
        config={
            "batch_size": batch_size,
            "max_epochs": max_epochs,
            "shapes": shapes,
            "learning_rate": repr(learning_rate),
        },
        dir=core.dirs["wandb"],
        append_to=callbacks,
        patch_tb=core.use_tb,
        patch_logdir=tb_dir,
    )

In [37]:
def get_u_net(
    input_shape, max_batch_size=32, activation=LeakyReLU(0.3), use_double=False
):
    nside = hp.npix2nside(input_shape[1])
    n_layers = math.floor(math.log(nside, 2))

    layers = []

    # Encoder path: progressively downsample and increase features
    encoder_channels = [2 ** (i + 5) for i in range(n_layers - 2)]
    n_encoder_layers = min(n_layers - 2, len(encoder_channels))

    for i in range(n_encoder_layers):
        fout = encoder_channels[i]

        # Convolutional blocks
        layers.append(
            HealpyChebyshev(
                K=3,
                Fout=fout,
                use_bn=True,
                use_bias=True,
                activation=activation,
            )
        )
        if use_double:
            layers.append(
                HealpyChebyshev(
                    K=3,
                    Fout=fout,
                    use_bn=True,
                    use_bias=True,
                    activation=activation,
                )
            )
        layers.append(Dropout(0.1))

        # Downsample
        layers.append(HealpyPool(1, "AVG"))

    # Bottleneck
    bottleneck_channels = encoder_channels[-1]
    layers.append(
        HealpyChebyshev(
            K=5,  # Larger receptive field at bottleneck
            Fout=bottleneck_channels,
            use_bn=True,
            use_bias=True,
            activation=activation,
        )
    )
    if use_double:
        layers.append(
            HealpyChebyshev(
                K=5,
                Fout=bottleneck_channels,
                use_bn=True,
                use_bias=True,
                activation=activation,
            )
        )
    layers.append(Dropout(0.1))

    # Decoder path: upsample and decrease features
    decoder_channels = encoder_channels[::-1][1:]  # Reverse and skip first

    for i, fout in enumerate(decoder_channels):
        # Upsample
        layers.append(HealpyPseudoConv_Transpose(1, fout))

        # Convolutional blocks
        layers.append(
            HealpyChebyshev(
                K=3,
                Fout=fout,
                use_bn=True,
                use_bias=True,
                activation=activation,
            )
        )
        if use_double:
            layers.append(
                HealpyChebyshev(
                    K=3,
                    Fout=fout,
                    use_bn=True,
                    use_bias=True,
                    activation=activation,
                )
            )
        layers.append(Dropout(0.1))

    # Final upsampling to original resolution
    layers.append(HealpyPseudoConv_Transpose(1, input_shape[-1]))

    # Final output layer - map to original number of channels
    layers.append(
        HealpyChebyshev(
            K=3,
            Fout=input_shape[-1],  # Same as input channels
            use_bn=False,
            use_bias=True,
            activation=None,  # Linear output for reconstruction
        )
    )

    model = HealpyGCNN(
        nside,
        indices=np.arange(input_shape[1]),
        layers=layers,
        n_neighbors=8,
        max_batch_size=max_batch_size,
        initial_Fin=input_shape[-1],
    )

    model.build(input_shape)
    return model

In [38]:
# ds = UnlensPhiMapDataset.fromCore(core, lensed=True)
# train, val, test = ds.split(
#     train_size=unet_split[0],
#     val_size=unet_split[1],
#     test_size=unet_split[2],
#     to_tf=True,
#     batch_size=batch_size,
#     duplicates=unet_duplicates,
#     # cache_file=unet_cache,
#     gen_batch_size=1,  # core.slurm.n_cpus,
# )

In [39]:
if not os.path.exists(unet_keras_file):
    logger.debug(f"U-Net file '{unet_keras_file}' does not exist, creating new U-Net")
    logger.info("Creating U-Net model and training data")

    # calculate the number of steps per epoch and decay steps for use later
    epoch_steps = math.ceil(
        core.total_sims * unet_split[0] * unet_duplicates[0] // batch_size
    )
    decay_steps = epoch_steps
    print("Steps per epoch:", epoch_steps)

    with strategy.scope():
        learning_rate = ExponentialDecay(initial_LR, decay_steps, 0.9, staircase=True)

        u_net = get_u_net((None, core.npix, core.npols), core.nside)

        u_net.compile(
            optimizer=AdamW(
                learning_rate, weight_decay=5e-6, amsgrad=True, use_ema=True
            ),
            loss="mse",  # RMSELoss(),
            metrics=rmse_metrics(shapes),
        )
else:
    logger.info("Loading U-Net from file: '%s'", unet_keras_file)
    u_net = tf.keras.models.load_model(unet_keras_file)

03-Nov-25 18:18:57 - mlpng.trainer - DEBUG - U-Net file 'data/models/l767_n256_T_10000_p1.0/unet-local-notebook-1762215536.keras' does not exist, creating new U-Net
03-Nov-25 18:18:57 - mlpng.trainer - INFO - Creating U-Net model and training data
03-Nov-25 18:18:57 - mlpng.trainer - INFO - Creating U-Net model and training data
Steps per epoch: 312


In [40]:
u_net.summary()

Model: "healpy_gcnn_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 chebyshev (Chebyshev)       (None, 786432, 32)        192       
                                                                 
 dropout (Dropout)           (None, 786432, 32)        0         
                                                                 
 healpy_pool (HealpyPool)    (None, 196608, 32)        0         
                                                                 
 chebyshev_1 (Chebyshev)     (None, 196608, 64)        6336      
                                                                 
 dropout_1 (Dropout)         (None, 196608, 64)        0         
                                                                 
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 chebyshev (Chebyshev)       (None, 786432, 32)      

In [41]:
# for x, y in train.take(1):
#     print(np.shape(x))
#     print(np.shape(y))

In [ ]:
ds = UnlensPhiMapDataset.fromCore(core, lensed=True)
train, val, test = ds.split(
    train_size=unet_split[0],
    val_size=unet_split[1],
    test_size=unet_split[2],
    to_tf=True,
    batch_size=batch_size,
    duplicates=unet_duplicates,
    # cache_file=unet_cache,
    gen_batch_size=1,  # core.slurm.n_cpus,
)

03-Nov-25 19:01:10 - mlpng.utils.dataloaders - DEBUG - Error information from 'data/data/l767_n256_T_10000_p1.0.hdf5'...
03-Nov-25 19:01:10 - mlpng.utils.dataloaders - DEBUG - Fisher Matrix: [0.00869114]
03-Nov-25 19:01:10 - mlpng.utils.dataloaders - DEBUG - Marginal Likelihoods: [10.726587]
03-Nov-25 19:01:10 - mlpng.utils.dataloaders - DEBUG - Fisher for shape local: 0.008432513, std div: 10.889839401764311
03-Nov-25 19:01:10 - mlpng.utils.dataloaders - DEBUG - Splitting 'data/data/l767_n256_T_10000_p1.0.hdf5' into train: 0:800 (800), val: 800:900 (100), test: 900:1000 (100)
03-Nov-25 19:01:10 - mlpng.utils.dataloaders - DEBUG - Fisher Matrix: [0.00869114]
03-Nov-25 19:01:10 - mlpng.utils.dataloaders - DEBUG - Marginal Likelihoods: [10.726587]
03-Nov-25 19:01:10 - mlpng.utils.dataloaders - DEBUG - Fisher for shape local: 0.008432513, std div: 10.889839401764311
03-Nov-25 19:01:10 - mlpng.utils.dataloaders - DEBUG - Splitting 'data/data/l767_n256_T_10000_p1.0.hdf5' into train: 0:800 (

In [42]:
# get this data that we will need later, also serves to create the test data
# this allows us to have the full cached dataset by the end of the first epoch
# if the cache exists this is very fast
if not os.path.exists(unet_keras_file):
    logger.debug("Creating test cache")
    y_test = np.concatenate([y for _, y in test])
    logger.debug("Test cache created")

03-Nov-25 19:01:11 - mlpng.trainer - DEBUG - Creating test cache


03-Nov-25 19:01:31 - mlpng.trainer - DEBUG - Test cache created


In [43]:
if not os.path.exists(unet_keras_file):
    logger.debug("Creating val cache")
    for _ in val:
        pass
    logger.debug("Val cache created")

03-Nov-25 19:01:31 - mlpng.trainer - DEBUG - Creating val cache


03-Nov-25 19:02:53 - mlpng.trainer - DEBUG - Val cache created


In [44]:
# just generate the training data for one epoch to fill the cache
# this might speed up training for the first epoch
if not os.path.exists(unet_keras_file):
    logger.debug("Creating train cache")
    for _ in train:
        pass
    logger.debug("Train cache created")

03-Nov-25 19:02:54 - mlpng.trainer - DEBUG - Creating train cache
03-Nov-25 19:30:08 - mlpng.trainer - DEBUG - Train cache created


In [ ]:
# and finally we fit the model
if not os.path.exists(unet_keras_file):
    history = u_net.fit(
        train,
        epochs=max_epochs,
        validation_data=val,
        callbacks=callbacks,
        verbose=1,
    )

Epoch 7/50


I0000 00:00:1762219826.344140 3204734 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


313/313 [==============================] - 703s 2s/step - loss: 0.0347 - rmse_local: 0.1634 - val_loss: 8.5346e-04 - val_rmse_local: 0.0197
Epoch 8/50
165/313 [==============>...............] - ETA: 5:11 - loss: 0.0257 - rmse_local: 0.1591

KeyboardInterrupt: 

## test save / load model

In [ ]:
if not os.path.exists(unet_keras_file):
    u_net.save(unet_keras_file)

    # just test loading the model
    with strategy.scope():
        u_net = tf.keras.models.load_model(unet_keras_file)

### test u_net

In [ ]:
# need to find a way to test the output of the model here
results = u_net.evaluate(test, return_dict=True)
logger.info("Test results: %s", results)

In [ ]:
# find a good method to test the u_net
# look at joels / eric paper for ideas
# plot powerspecturm and redisuals from the test set

test_phi = np.array([y for _, y in test])
test_phi = test_phi.reshape(-1, *test_phi.shape[-2:])
pred_phi = u_net.predict(test)

for map, name in [(test_phi[0, :, 0], "True"), (pred_phi[0, :, 0], "Predicted")]:
    r_map = hp.reorder(map, n2r=True)
    hp.mollview(r_map, title=f"{name} phi map", unit="phi", cmap="viridis")
    plt.show()

ells = np.arange(core.lmax + 1)
ell_scale = ells * (ells + 1) / (2 * np.pi)
plt.figure()
for map, name in [(test_phi[0, :, 0], "True"), (pred_phi[0, :, 0], "Predicted")]:
    r_map = hp.reorder(map, n2r=True)
    cl = hp.anafast(r_map, lmax=core.lmax, pol=False, use_pixel_weights=True)
    plt.loglog(cl * ell_scale, label=name)

plt.legend()
plt.xlabel(r"$\ell$")
plt.ylabel(r"$(\ell * (\ell + 1) / (2 \pi) C_\ell^{\phi\phi})$")
plt.show()

# make fnl model based on new data

In [ ]:
from mlpng.scn_jorik import get_model as get_fnl_model

ds_2 = MapDataset.fromCore(core, lensed=False)
train_2, val_2, test_2 = ds_2.split(
    train_size=fnl_split[0],
    val_size=fnl_split[1],
    test_size=fnl_split[2],
    to_tf=True,
    batch_size=batch_size,
    duplicates=fnl_duplicates,
    cache_file=fnl_cache,
    gen_batch_size=core.slurm.n_cpus,
)

if not os.path.exists(fnl_keras_file):
    # the paper does not rotate the test set
    test_2.rotate = False

    logger.debug("Creating fnl caches")
    for _ in train_2:
        pass
    logger.debug("Fnl train cache created")
    for _ in val_2:
        pass
    logger.debug("Fnl val cache created")
    for _ in test_2:
        pass
    logger.debug("Fnl caches created")

    # calculate the number of steps per epoch and decay steps for use later
    epoch_steps = math.ceil(
        core.total_sims * fnl_split[0] * fnl_duplicates[0] // batch_size
    )
    decay_steps = epoch_steps * 3

    with strategy.scope():
        learning_rate = ExponentialDecay(initial_LR, decay_steps, 0.95, staircase=True)

        fnl_model = get_fnl_model(
            (None, core.npix, core.npols), batch_size, len(shapes)
        )

        fnl_model.compile(
            optimizer=AdamW(learning_rate),
            loss=RMSELoss(),
            metrics=rmse_metrics(shapes),
        )

    fnl_model.summary()

    # and finally we fit the model
    fnl_history = fnl_model.fit(
        train_2,
        epochs=max_epochs,
        validation_data=val_2,
        callbacks=callbacks,
        verbose=1,
    )

    logger.info("Saving model to %s", fnl_keras_file)
    fnl_model.save(fnl_keras_file)
else:
    logger.info("Loading model from %s", fnl_keras_file)
    fnl_model = tf.keras.models.load_model(fnl_keras_file)

In [ ]:
# plot the test results for model 2, make sure it does not pass the KR bound

# Check Melsen for the expected bounds, reproduce results
# | nside | Full | Gaussian |
# |------:|-----:|---------:|
# | 32    |   95 |      103 |
# | 64    |   56 |       52 |
# | 128   |   33 |       28 |

In [ ]:
fnl_2_truth = np.array([y for _, y in test_2])
fnl_2_preds = fnl_model.predict(test_2, verbose=0)

In [ ]:
vals = fnl_2_truth.ravel()
plt.figure(figsize=(6, 4))
plt.hist(vals, bins=50, color="C0", alpha=0.8)
plt.xlabel("fnl (truth)")
plt.ylabel("Count")
plt.title(f"Distribution of test_2 truth values (N={vals.size})")
plt.tight_layout()
plt.show()

In [ ]:
fnl_2_truth = np.array(fnl_2_truth).reshape(-1, 1)

# Calculate metrics
fnl_error = fnl_2_preds - fnl_2_truth
print(f"Mean absolute error: {np.mean(np.abs(fnl_error)):.4f}")
print(f"RMSE: {np.sqrt(np.mean(fnl_error**2)):.4f}")

line = np.array([np.nanmin(fnl_2_truth), np.nanmax(fnl_2_truth)])
sigma = core.get_likelihoods(False)[0]

# Plot results
plt.figure(figsize=(16, 4))

plt.subplot(131)
plt.scatter(fnl_2_truth, fnl_2_preds, alpha=0.5)
plt.plot(line, line, "r--")
plt.plot(line, line + sigma, "g--")
plt.plot(line, line - sigma, "g--")
plt.xlabel("True fnl")
plt.ylabel("Predicted fnl")
plt.title("fnl Prediction from unlensed Maps")

plt.subplot(132)
plt.hist(fnl_error, bins=50)
plt.axvline(sigma, color="g", linestyle="--", label=f"+σ={sigma:.2f}")
plt.axvline(-sigma, color="g", linestyle="--", label=f"-σ={-sigma:.2f}")
plt.xlabel("Prediction Error")
plt.ylabel("Count")
plt.title("fnl Error Distribution")

plt.subplot(133)
plt.plot(fnl_2_preds, alpha=0.7, label="Predicted")
plt.plot(fnl_2_truth, alpha=0.7, label="True")
plt.xlabel("Sample Index")
plt.ylabel("fnl Value")
plt.title("fnl Values Over Test Set")
plt.legend()

plt.tight_layout()
plt.show()

# delens maps to connect the models

In [ ]:
ds_3 = MapDataset.fromCore(core, lensed=True)
train_3, val_3, test_3 = ds_3.split(
    train_size=final_split[0],
    val_size=final_split[1],
    test_size=final_split[2],
    to_tf=True,
    batch_size=batch_size,
    duplicates=final_duplicates,
    cache_file=final_cache,
    gen_batch_size=core.slurm.n_cpus,
)

print("Creating test_3 cache")
fnl_truth = np.concatenate([y for _, y in test_3])
print("Done")

In [ ]:
from pixell import reproject, lensing, enmap, utils
import numpy as np
import tensorflow as tf

res_arcmin = hp.nside2resol(core.nside, arcmin=True)
res_rad = res_arcmin * utils.arcmin

# snap to nearest resolution that evenly divides the sky (vertical)
ny = int(round(np.pi / res_rad))
res_fixed = np.pi / ny  # exact divisor for π
print(
    f"healpy res (arcmin) = {res_arcmin:.6f}, using fixed res (arcmin) = {res_fixed / utils.arcmin:.6f}"
)

# build geometry with the fixed resolution
shape, wcs = enmap.fullsky_geometry(res_fixed)


def _delens_py(lensed, phi, nside=core.nside):
    delensed_maps = np.zeros_like(lensed)
    # TODO test joblib parallel here
    for i, (l_map, phi) in enumerate(zip(lensed, phi)):
        lensed_enmap = reproject.healpix2map(np.asarray(l_map).T, shape, wcs)
        phi_enmap = reproject.healpix2map(np.asarray(phi).T, shape, wcs)
        delensed_enmap = lensing.delens_map(lensed_enmap, phi_enmap)

        delensed_maps[i] = reproject.map2healpix(delensed_enmap, nside).T

    return delensed_maps


def _map_fn(pair, phi):
    lensed, truth = pair
    delensed = tf.py_function(
        func=_delens_py,
        inp=[lensed, phi],
        Tout=tf.float32,
    )

    delensed.set_shape([None, core.npix, core.npols])
    truth.set_shape([None, len(shapes)])

    # delensed.set_shape([core.npix, core.npols])
    return delensed, truth


phi_preds = u_net.predict(test_3, verbose=0)

phi_ds = tf.data.Dataset.from_tensor_slices(phi_preds).batch(batch_size)
delensed_ds = (
    tf.data.Dataset.zip((test_3, phi_ds))
    .map(_map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()  # just cache here to ensure no mixing of the datasets on future reads
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
for map, _ in delensed_ds.take(1):
    print("Delensed map shape:", map.shape)
    hp.mollview(
        hp.reorder(map[0, :, 0].numpy(), n2r=True),
        title="Delensed map",
        unit="T",
        cmap="viridis",
    )
    plt.show()

In [ ]:
## TODO test speed of this vs joblib parallel version

In [ ]:
fnl_preds = fnl_model.predict(delensed_ds, verbose=1)

In [ ]:
fnl_truth = np.array([y for _, y in delensed_ds])

In [ ]:
fnl_truth = fnl_truth.ravel()
fnl_preds = fnl_preds.ravel()

print("True fnls shape:", fnl_truth.shape)
print("Predicted fnls shape:", fnl_preds.shape)

# Calculate metrics
fnl_error = fnl_preds - fnl_truth
print(f"Mean absolute error: {np.mean(np.abs(fnl_error)):.4f}")
print(f"RMSE: {np.sqrt(np.mean(fnl_error**2)):.4f}")

line = np.array([np.nanmin(fnl_truth), np.nanmax(fnl_truth)])
sigma = core.get_likelihoods(True)[0]
# Plot results
plt.figure(figsize=(12, 4))

plt.subplot(131)
plt.scatter(fnl_truth, fnl_preds, alpha=0.5)
plt.plot(line, line, "r--")
plt.plot(line, line + sigma, "g--")
plt.plot(line, line - sigma, "g--")
plt.xlabel("True fnl")
plt.ylabel("Predicted fnl")
plt.title("fnl Prediction from Delensed Maps")

plt.subplot(132)
plt.hist(fnl_error, bins=50)
plt.axvline(sigma, color="g", linestyle="--", label=f"+σ={sigma:.2f}")
plt.axvline(-sigma, color="g", linestyle="--", label=f"-σ={-sigma:.2f}")
plt.xlabel("Prediction Error")
plt.ylabel("Count")
plt.title("fnl Error Distribution")

plt.subplot(133)
plt.plot(fnl_preds, alpha=0.7, label="Predicted")
plt.plot(fnl_truth, alpha=0.7, label="True")
plt.xlabel("Sample Index")
plt.ylabel("fnl Value")
plt.title("fnl Values Over Test Set")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
vals = fnl_truth.ravel()  # assumes fnl_truth exists from earlier cell

plt.figure(figsize=(6, 4))
plt.hist(vals, bins=50, color="C0", alpha=0.8)
plt.xlabel("fnl (truth)")
plt.ylabel("Count")
plt.title(f"Distribution of test_3 truth values (N={vals.size})")
plt.tight_layout()
plt.show()